In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('/content/NirmaanAI_Feature_Engineered.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (182, 40)

First 5 rows:


,Sr. No.,Sector Name,Line Ministry,Implementing Agency,Project Code,Project Name,Original Cost\n(in cr.),Revised Cost\n(in cr.),Expenditure\n(in cr.),Physical Progress\n(in %),...,Progress Deviation,Progress Status,Remaining Project Time Days,Remaining Project Time Months,Planned Duration Years,Time Elapsed Years,Schedule Delay Years,Remaining Project Time Years,Delay Risk,Delay Risk Label
0,1,Aviation & Aviation Infrastructure,Ministry of Civil Aviation,Airport Authority of India [AAI],611570,SITC of ATM Automation System and ASMGCS [Incl...,500,NaN,399.77,80,...,-20.000000,Behind Schedule,-5.0,-0.166667,4.73,5.71,0.97,-0.01,1.0,Delayed
1,2,Coal,Ministry of Coal,WCL - CIL,616223,ADASA UG TO OC MINE,316,NaN,209.29,86,...,-14.000000,Behind Schedule,207.0,6.900000,5.37,7.80,3.00,0.57,1.0,Delayed
2,3,Coal,Ministry of Coal,WCL - CIL,616229,AMAL GONDEGAON GHATROHANA EXPN,669,NaN,2.95,3,...,-40.595911,Severely Behind,NaN,NaN,4.55,1.98,NaN,NaN,NaN,NaN
3,4,Coal,Ministry of Coal,WCL - CIL,616050,AMALGAMATED INDER-KAMPTEE DEEP,236,NaN,128.49,99,...,-1.000000,On Track,207.0,6.900000,5.52,7.95,3.00,0.57,1.0,Delayed
4,5,Coal,Ministry of Coal,WCL - CIL,400177,AMALGAMATED YEKONA I & YEKONA,727,NaN,564.25,81,...,-19.000000,Behind Schedule,207.0,6.900000,9.63,11.07,2.00,0.57,1.0,Delayed


In [4]:
# ============================================
# NirmaanAI - ML Model
# Step 2: Separate Labeled and Unlabeled Data
# ============================================

# Projects with known delay outcome
labeled_df = df[df['Delay Risk'].notna()].copy()

# Projects with unknown delay outcome
unlabeled_df = df[df['Delay Risk'].isna()].copy()

print("Labeled projects:", len(labeled_df))
print("Unlabeled / ongoing projects:", len(unlabeled_df))

print("\nDelay Risk distribution:")
print(labeled_df['Delay Risk'].value_counts())

print("\nExpected:")
print("Labeled = 134")
print("Unlabeled = 48")

Labeled projects: 134
Unlabeled / ongoing projects: 48

Delay Risk distribution:
Delay Risk
1.0    105
0.0     29
Name: count, dtype: int64

Expected:
Labeled = 134
Unlabeled = 48


In [5]:
# ============================================
# NirmaanAI - ML Model
# Step 3: Select Features and Target
# ============================================

ml_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Time Elapsed Days',
    'Time Elapsed %',
    'Expected Progress %',
    'Progress Deviation',
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

target = 'Delay Risk'

# Create X and y using only labeled projects
X = labeled_df[ml_features].copy()
y = labeled_df[target].astype(int)

print("Features selected:", len(ml_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
for feature in ml_features:
    print("-", feature)

print("\nTarget distribution:")
print(y.value_counts())

Features selected: 14
X shape: (134, 14)
y shape: (134,)

Features:
- Original Cost
(in cr.)
- Expenditure
(in cr.)
- Physical Progress
(in %)
- Expenditure %
- Remaining Budget
(in cr.)
- Progress Gap
- Planned Duration Days
- Time Elapsed Days
- Time Elapsed %
- Expected Progress %
- Progress Deviation
- Sector Name
- Line Ministry
- Implementing Agency

Target distribution:
Delay Risk
1    105
0     29
Name: count, dtype: int64


In [6]:
# ============================================
# NirmaanAI - ML Model
# Step 4: Train-Test Split
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (107, 14)
Testing set: (27, 14)

Training target distribution:
Delay Risk
1    84
0    23
Name: count, dtype: int64

Testing target distribution:
Delay Risk
1    21
0     6
Name: count, dtype: int64


In [7]:
# ============================================
# NirmaanAI - ML Model
# Step 5A: Inspect Features Before Preprocessing
# ============================================

print("Feature data types:\n")
print(X.dtypes)

print("\n" + "="*50)

print("\nMissing values in training features:\n")
print(X_train.isnull().sum())

Feature data types:

Original Cost\n(in cr.)         int64
Expenditure\n(in cr.)         float64
Physical Progress\n(in %)       int64
Expenditure %                 float64
Remaining Budget\n(in cr.)    float64
Progress Gap                  float64
Planned Duration Days         float64
Time Elapsed Days               int64
Time Elapsed %                float64
Expected Progress %           float64
Progress Deviation            float64
Sector Name                    object
Line Ministry                  object
Implementing Agency            object
dtype: object


Missing values in training features:

Original Cost\n(in cr.)       0
Expenditure\n(in cr.)         0
Physical Progress\n(in %)     0
Expenditure %                 0
Remaining Budget\n(in cr.)    0
Progress Gap                  0
Planned Duration Days         0
Time Elapsed Days             0
Time Elapsed %                0
Expected Progress %           0
Progress Deviation            0
Sector Name                   0
Line Mini

In [8]:
# ============================================
# NirmaanAI - ML Model
# Step 5B: Build Preprocessing Pipeline
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Separate numerical and categorical features
numeric_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Time Elapsed Days',
    'Time Elapsed %',
    'Expected Progress %',
    'Progress Deviation'
]

categorical_features = [
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")
print("\nNumerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully!

Numerical features: 11
Categorical features: 3


In [9]:
# ============================================
# NirmaanAI - ML Model
# Step 6A: Logistic Regression Baseline
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create complete ML pipeline
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# Train the model
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [10]:
# ============================================
# NirmaanAI - ML Model
# Step 6B: Evaluate Logistic Regression
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions on unseen test data
y_pred = logistic_pipeline.predict(X_test)
y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("===== Logistic Regression Results =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== Classification Report =====")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Not Delayed', 'Delayed'],
    zero_division=0
))

===== Logistic Regression Results =====
Accuracy : 0.8148
Precision: 1.0000
Recall   : 0.7619
F1 Score : 0.8649
ROC-AUC  : 0.9921

===== Confusion Matrix =====
[[ 6  0]
 [ 5 16]]

===== Classification Report =====
              precision    recall  f1-score   support

 Not Delayed       0.55      1.00      0.71         6
     Delayed       1.00      0.76      0.86        21

    accuracy                           0.81        27
   macro avg       0.77      0.88      0.79        27
weighted avg       0.90      0.81      0.83        27



In [11]:
# ============================================
# NirmaanAI - ML Model
# Step 7A: Random Forest
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Create Random Forest pipeline
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
random_forest_pipeline.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [12]:
# ============================================
# NirmaanAI - ML Model
# Step 7B: Evaluate Random Forest
# ============================================

# Predictions on unseen test data
rf_pred = random_forest_pipeline.predict(X_test)
rf_prob = random_forest_pipeline.predict_proba(X_test)[:, 1]

# Calculate metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, zero_division=0)
rf_recall = recall_score(y_test, rf_pred, zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, zero_division=0)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("===== Random Forest Results =====")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_test, rf_pred))

print("\n===== Classification Report =====")
print(classification_report(
    y_test,
    rf_pred,
    target_names=['Not Delayed', 'Delayed'],
    zero_division=0
))

===== Random Forest Results =====
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

===== Confusion Matrix =====
[[ 6  0]
 [ 0 21]]

===== Classification Report =====
              precision    recall  f1-score   support

 Not Delayed       1.00      1.00      1.00         6
     Delayed       1.00      1.00      1.00        21

    accuracy                           1.00        27
   macro avg       1.00      1.00      1.00        27
weighted avg       1.00      1.00      1.00        27



In [13]:
# ============================================
# NirmaanAI - ML Model
# Step 7C: Random Forest Feature Importance
# ============================================

# Get the trained preprocessing step
rf_preprocessor = random_forest_pipeline.named_steps['preprocessor']

# Get the trained Random Forest model
rf_model = random_forest_pipeline.named_steps['model']

# Get feature names after preprocessing
feature_names = rf_preprocessor.get_feature_names_out()

# Get feature importance
importances = rf_model.feature_importances_

# Create importance table
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 20 features used by Random Forest:\n")
display(feature_importance_df.head(20))

Top 20 features used by Random Forest:



,Feature,Importance
8,num__Time Elapsed %,0.205457
9,num__Expected Progress %,0.178663
7,num__Time Elapsed Days,0.099053
2,num__Physical Progress\n(in %),0.078084
1,num__Expenditure\n(in cr.),0.066764
0,num__Original Cost\n(in cr.),0.057198
4,num__Remaining Budget\n(in cr.),0.050191
10,num__Progress Deviation,0.048027
6,num__Planned Duration Days,0.045846
3,num__Expenditure %,0.044688


In [14]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models we have built so far
models = {
    'Logistic Regression': logistic_pipeline,
    'Random Forest': random_forest_pipeline
}

# Metrics to evaluate
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = []

for model_name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    cv_results.append({
        'Model': model_name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.8697,0.9497,0.8801,0.9123,0.8913
1,Random Forest,0.8429,0.8917,0.9176,0.9016,0.9021


In [15]:
# Test the trained Random Forest model on a completely new project

new_project = pd.DataFrame([{
    'Original Cost\n(in cr.)': 500,
    'Expenditure\n(in cr.)': 320,
    'Physical Progress\n(in %)': 45,
    'Expenditure %': 64,
    'Remaining Budget\n(in cr.)': 180,
    'Progress Gap': 19,
    'Planned Duration Days': 1200,
    'Time Elapsed Days': 1050,
    'Time Elapsed %': 87.5,
    'Expected Progress %': 87.5,
    'Progress Deviation': -42.5,
    'Sector Name': 'Road Transport',
    'Line Ministry': 'Ministry of Road Transport',
    'Implementing Agency': 'NHAI'
}])

# Prediction
prediction = random_forest_pipeline.predict(new_project)[0]

# Probability
probability = random_forest_pipeline.predict_proba(new_project)[0][1]

print("Prediction:", "DELAYED" if prediction == 1 else "NOT DELAYED")
print(f"Delay Probability: {probability:.2%}")

Prediction: DELAYED
Delay Probability: 55.67%


In [16]:
# Test the Random Forest model on 3 hypothetical projects

test_projects = pd.DataFrame([
    {
        # 🟢 PROJECT A — Low Risk / On Track
        'Original Cost\n(in cr.)': 500,
        'Expenditure\n(in cr.)': 200,
        'Physical Progress\n(in %)': 70,
        'Expenditure %': 40,
        'Remaining Budget\n(in cr.)': 300,
        'Progress Gap': -5,
        'Planned Duration Days': 1200,
        'Time Elapsed Days': 800,
        'Time Elapsed %': 66.7,
        'Expected Progress %': 66.7,
        'Progress Deviation': 3.3,
        'Sector Name': 'Road Transport',
        'Line Ministry': 'Ministry of Road Transport',
        'Implementing Agency': 'NHAI'
    },
    {
        # 🟡 PROJECT B — Medium Risk / Warning Signs
        'Original Cost\n(in cr.)': 500,
        'Expenditure\n(in cr.)': 300,
        'Physical Progress\n(in %)': 45,
        'Expenditure %': 60,
        'Remaining Budget\n(in cr.)': 200,
        'Progress Gap': 15,
        'Planned Duration Days': 1200,
        'Time Elapsed Days': 850,
        'Time Elapsed %': 70.8,
        'Expected Progress %': 70.8,
        'Progress Deviation': -25.8,
        'Sector Name': 'Road Transport',
        'Line Ministry': 'Ministry of Road Transport',
        'Implementing Agency': 'NHAI'
    },
    {
        # 🔴 PROJECT C — High Risk / Severe Delay Indicators
        'Original Cost\n(in cr.)': 500,
        'Expenditure\n(in cr.)': 450,
        'Physical Progress\n(in %)': 30,
        'Expenditure %': 90,
        'Remaining Budget\n(in cr.)': 50,
        'Progress Gap': 60,
        'Planned Duration Days': 1000,
        'Time Elapsed Days': 1100,
        'Time Elapsed %': 110,
        'Expected Progress %': 100,
        'Progress Deviation': -70,
        'Sector Name': 'Road Transport',
        'Line Ministry': 'Ministry of Road Transport',
        'Implementing Agency': 'NHAI'
    }
])

# Generate predictions
predictions = random_forest_pipeline.predict(test_projects)
probabilities = random_forest_pipeline.predict_proba(test_projects)[:, 1]

# Display results
for i, (prediction, probability) in enumerate(
    zip(predictions, probabilities), start=1
):
    print(f"Project {i}:")
    print("Prediction:", "🔴 DELAYED" if prediction == 1 else "🟢 NOT DELAYED")
    print(f"Delay Probability: {probability:.2%}")
    print("-" * 40)

Project 1:
Prediction: 🟢 NOT DELAYED
Delay Probability: 40.67%
----------------------------------------
Project 2:
Prediction: 🟢 NOT DELAYED
Delay Probability: 35.33%
----------------------------------------
Project 3:
Prediction: 🔴 DELAYED
Delay Probability: 92.67%
----------------------------------------


In [17]:
   # ==========================================
# TEMPORAL LEAKAGE AUDIT
# ==========================================

current_date = pd.Timestamp('2026-09-05')

# Convert dates safely
labeled_df['Original_Date'] = pd.to_datetime(
    labeled_df['Original\nDate of Commissioning'],
    errors='coerce'
)

labeled_df['Revised_Date'] = pd.to_datetime(
    labeled_df['Revised\nDate of Commissioning'],
    errors='coerce'
)

# Check whether the revised completion date has already passed
labeled_df['Outcome_Already_Passed'] = (
    labeled_df['Revised_Date'] < current_date
)

# Summary
total_labeled = len(labeled_df)
known_revised_dates = labeled_df['Revised_Date'].notna().sum()
outcome_already_passed = labeled_df['Outcome_Already_Passed'].sum()

print("TEMPORAL LEAKAGE AUDIT")
print("=" * 45)

print(f"Total labeled projects: {total_labeled}")
print(f"Projects with revised completion date: {known_revised_dates}")
print(f"Projects whose revised date has already passed: {outcome_already_passed}")

print("\nPercentage whose revised date has already passed:")
print(f"{(outcome_already_passed / total_labeled) * 100:.2f}%")

# Check current-state features
temporal_features = [
    'Physical Progress\n(in %)',
    'Time Elapsed Days',
    'Time Elapsed %',
    'Expected Progress %',
    'Progress Deviation'
]

print("\nCurrent-state features being used:")
for feature in temporal_features:
    print("•", feature)

TEMPORAL LEAKAGE AUDIT
Total labeled projects: 134
Projects with revised completion date: 134
Projects whose revised date has already passed: 47

Percentage whose revised date has already passed:
35.07%

Current-state features being used:
• Physical Progress
(in %)
• Time Elapsed Days
• Time Elapsed %
• Expected Progress %
• Progress Deviation


In [18]:
# ==========================================
# CURRENT-STATE FEATURE AUDIT
# ==========================================

audit_features = [
    'Physical Progress\n(in %)',
    'Time Elapsed Days',
    'Time Elapsed %',
    'Expected Progress %',
    'Progress Deviation'
]

print("CURRENT-STATE FEATURE AUDIT")
print("=" * 50)

# Compare feature averages between delayed and non-delayed projects
for feature in audit_features:
    delayed_mean = labeled_df.loc[
        labeled_df['Delay Risk'] == 1, feature
    ].mean()

    not_delayed_mean = labeled_df.loc[
        labeled_df['Delay Risk'] == 0, feature
    ].mean()

    print(f"\n{feature}")
    print(f"  Delayed projects     : {delayed_mean:.2f}")
    print(f"  Not Delayed projects : {not_delayed_mean:.2f}")

print("\n" + "=" * 50)
print("Interpretation:")
print("Large differences are not automatically leakage.")
print("They indicate that the feature strongly distinguishes the two classes.")

CURRENT-STATE FEATURE AUDIT

Physical Progress
(in %)
  Delayed projects     : 77.88
  Not Delayed projects : 46.07

Time Elapsed Days
  Delayed projects     : 2690.96
  Not Delayed projects : 1748.14

Time Elapsed %
  Delayed projects     : 174.56
  Not Delayed projects : 78.85

Expected Progress %
  Delayed projects     : 98.39
  Not Delayed projects : 76.77

Progress Deviation
  Delayed projects     : -20.51
  Not Delayed projects : -30.70

Interpretation:
Large differences are not automatically leakage.
They indicate that the feature strongly distinguishes the two classes.


In [19]:
# Install XGBoost if needed
!pip install -q xgboost

from xgboost import XGBClassifier

In [20]:
# ==========================================
# XGBOOST PIPELINE
# ==========================================

from sklearn.pipeline import Pipeline

# Calculate class imbalance from TRAINING data only
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("XGBoost class weight:", round(scale_pos_weight, 2))

# Create XGBoost model
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Use the SAME preprocessing as Logistic Regression and Random Forest
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

print("\nXGBoost pipeline created successfully!")

XGBoost class weight: 0.27

XGBoost pipeline created successfully!


In [21]:
# ==========================================
# TRAIN XGBOOST
# ==========================================

xgb_pipeline.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [22]:
# ==========================================
# 5-FOLD CROSS-VALIDATION — XGBOOST
# ==========================================

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

xgb_cv_scores = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

print("XGBoost Cross-Validation Results:")
print("----------------------------------")
print("Accuracy :", round(xgb_cv_scores['test_accuracy'].mean(), 4))
print("Precision:", round(xgb_cv_scores['test_precision'].mean(), 4))
print("Recall   :", round(xgb_cv_scores['test_recall'].mean(), 4))
print("F1 Score :", round(xgb_cv_scores['test_f1'].mean(), 4))
print("ROC-AUC  :", round(xgb_cv_scores['test_roc_auc'].mean(), 4))

XGBoost Cross-Validation Results:
----------------------------------
Accuracy : 0.8797
Precision: 0.9616
Recall   : 0.8824
F1 Score : 0.9185
ROC-AUC  : 0.9215


In [23]:
# ==========================================
# FINAL TEST EVALUATION — XGBOOST
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Make predictions on untouched test data
y_test_pred = xgb_pipeline.predict(X_test)
y_test_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

print("XGBoost — Final Test Results")
print("----------------------------------")

print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision:", round(precision_score(y_test, y_test_pred), 4))
print("Recall   :", round(recall_score(y_test, y_test_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_test_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Not Delayed", "Delayed"]
))

XGBoost — Final Test Results
----------------------------------
Accuracy : 0.8889
Precision: 1.0
Recall   : 0.8571
F1 Score : 0.9231
ROC-AUC  : 1.0

Confusion Matrix:
[[ 6  0]
 [ 3 18]]

Classification Report:
              precision    recall  f1-score   support

 Not Delayed       0.67      1.00      0.80         6
     Delayed       1.00      0.86      0.92        21

    accuracy                           0.89        27
   macro avg       0.83      0.93      0.86        27
weighted avg       0.93      0.89      0.90        27



In [24]:
# ==========================================
# SAVE TRAINED XGBOOST PIPELINE
# ==========================================

import joblib

model_path = '/content/NirmaanAI_XGBoost_Delay_Model.pkl'

joblib.dump(xgb_pipeline, model_path)

print("XGBoost model saved successfully!")
print("Saved at:", model_path)

XGBoost model saved successfully!
Saved at: /content/NirmaanAI_XGBoost_Delay_Model.pkl


In [ ]:
# ==========================================
# SAVE ALL CLASSIFICATION MODELS
# ==========================================

import joblib

# Save Logistic Regression
joblib.dump(
    logistic_pipeline,
    '/content/NirmaanAI_Logistic_Regression.pkl'
)

# Save Random Forest
joblib.dump(
    random_forest_pipeline,
    '/content/NirmaanAI_Random_Forest.pkl'
)

# Save XGBoost
joblib.dump(
    xgb_pipeline,
    '/content/NirmaanAI_XGBoost_Delay_Model.pkl'
)

print("All classification models saved successfully!")

In [25]:
# Save Logistic Regression
joblib.dump(
    logistic_pipeline,
    '/content/NirmaanAI_Logistic_Regression.pkl'
)

# Save Random Forest
joblib.dump(
    random_forest_pipeline,
    '/content/NirmaanAI_Random_Forest.pkl'
)

print("Logistic Regression and Random Forest saved successfully!")

Logistic Regression and Random Forest saved successfully!


In [26]:
delay_duration = df['Schedule Delay Months']

print("Total projects:", len(df))
print("Projects with known delay duration:", delay_duration.notna().sum())
print("Missing delay duration:", delay_duration.isna().sum())

print("\nDelay duration statistics:")
print(delay_duration.describe())

print("\nNumber of projects with positive delay:")
print((delay_duration > 0).sum())

print("\nNumber of projects with zero/negative delay:")
print((delay_duration <= 0).sum())

Total projects: 182
Projects with known delay duration: 134
Missing delay duration: 48

Delay duration statistics:
count    134.000000
mean      24.801244
std       28.293398
min      -53.800000
25%        2.108333
50%       19.150000
75%       35.250000
max      103.466667
Name: Schedule Delay Months, dtype: float64

Number of projects with positive delay:
105

Number of projects with zero/negative delay:
29


In [27]:
# ==========================================
# PREPARE DELAY DURATION REGRESSION DATA
# ==========================================

# Use only projects with a known delay duration
reg_df = df[df['Schedule Delay Months'].notna()].copy()

# Features
X_reg = reg_df[ml_features].copy()

# Target
y_reg = reg_df['Schedule Delay Months'].copy()

print("Regression dataset shape:", X_reg.shape)
print("Target shape:", y_reg.shape)

print("\nTarget summary:")
print(y_reg.describe())

Regression dataset shape: (134, 14)
Target shape: (134,)

Target summary:
count    134.000000
mean      24.801244
std       28.293398
min      -53.800000
25%        2.108333
50%       19.150000
75%       35.250000
max      103.466667
Name: Schedule Delay Months, dtype: float64


In [28]:
# ==========================================
# TRAIN / TEST SPLIT — REGRESSION
# ==========================================

from sklearn.model_selection import train_test_split

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_reg_train.shape[0])
print("Testing samples :", X_reg_test.shape[0])

print("\nTraining target mean:", round(y_reg_train.mean(), 2))
print("Testing target mean :", round(y_reg_test.mean(), 2))

Training samples: 107
Testing samples : 27

Training target mean: 26.55
Testing target mean : 17.86


In [29]:
# ==========================================
# CHECK REGRESSION TARGET DISTRIBUTION
# ==========================================

print("Training target:")
print(y_reg_train.describe())

print("\nTesting target:")
print(y_reg_test.describe())

Training target:
count    107.000000
mean      26.552960
std       29.685407
min      -53.800000
25%        2.550000
50%       20.300000
75%       37.450000
max      103.466667
Name: Schedule Delay Months, dtype: float64

Testing target:
count    27.000000
mean     17.859259
std      20.966555
min     -12.133333
25%       0.883333
50%      12.166667
75%      28.283333
max      70.266667
Name: Schedule Delay Months, dtype: float64


In [30]:
# ==========================================
# REGRESSION MODELS
# ==========================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

# Linear Regression
linear_regression_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Random Forest Regressor
random_forest_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# XGBoost Regressor
xgb_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42,
        n_jobs=-1
    ))
])

print("All three regression pipelines created successfully!")

All three regression pipelines created successfully!


In [31]:
# ==========================================
# TRAIN ALL REGRESSION MODELS
# ==========================================

print("Training Linear Regression...")
linear_regression_pipeline.fit(X_reg_train, y_reg_train)

print("Training Random Forest Regressor...")
random_forest_reg_pipeline.fit(X_reg_train, y_reg_train)

print("Training XGBoost Regressor...")
xgb_reg_pipeline.fit(X_reg_train, y_reg_train)

print("\nAll three regression models trained successfully!")

Training Linear Regression...
Training Random Forest Regressor...
Training XGBoost Regressor...

All three regression models trained successfully!


In [32]:
# ==========================================
# 5-FOLD CROSS-VALIDATION — REGRESSION
# ==========================================

from sklearn.model_selection import KFold, cross_validate

# 5-fold CV
reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

reg_scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}

reg_models = {
    'Linear Regression': linear_regression_pipeline,
    'Random Forest Regressor': random_forest_reg_pipeline,
    'XGBoost Regressor': xgb_reg_pipeline
}

reg_results = {}

for name, model in reg_models.items():

    scores = cross_validate(
        model,
        X_reg_train,
        y_reg_train,
        cv=reg_cv,
        scoring=reg_scoring
    )

    reg_results[name] = {
        'MAE': -scores['test_mae'].mean(),
        'RMSE': -scores['test_rmse'].mean(),
        'R2': scores['test_r2'].mean()
    }

print("Regression Cross-Validation Results")
print("====================================")

for name, metrics in reg_results.items():
    print(f"\n{name}")
    print("MAE :", round(metrics['MAE'], 4), "months")
    print("RMSE:", round(metrics['RMSE'], 4), "months")
    print("R²  :", round(metrics['R2'], 4))

Regression Cross-Validation Results

Linear Regression
MAE : 11.3522 months
RMSE: 28.3638 months
R²  : -0.1098

Random Forest Regressor
MAE : 7.6256 months
RMSE: 13.8183 months
R²  : 0.7385

XGBoost Regressor
MAE : 8.4561 months
RMSE: 15.0238 months
R²  : 0.7005


In [34]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Parameter combinations to search
rf_param_grid = {
    'model__n_estimators': randint(200, 800),
    'model__max_depth': [None, 5, 8, 10, 15, 20, 25],
    'model__min_samples_split': randint(2, 10),
    'model__min_samples_leaf': randint(1, 5),
    'model__max_features': ['sqrt', 'log2', 0.7, 1.0]
}

# Randomized search
rf_random_search = RandomizedSearchCV(
    estimator=random_forest_reg_pipeline,
    param_distributions=rf_param_grid,
    n_iter=40,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit search on TRAINING data only
rf_random_search.fit(X_reg_train, y_reg_train)

print("\nBest Parameters:")
print(rf_random_search.best_params_)

print("\nBest Cross-Validation MAE:")
print(round(-rf_random_search.best_score_, 4), "months")

Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best Parameters:
{'model__max_depth': None, 'model__max_features': 0.7, 'model__min_samples_leaf': 1, 'model__min_samples_split': 3, 'model__n_estimators': 430}

Best Cross-Validation MAE:
8.2339 months


In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions
rf_train_pred = random_forest_reg_pipeline.predict(X_reg_train)
rf_test_pred = random_forest_reg_pipeline.predict(X_reg_test)

# Metrics
print("RANDOM FOREST REGRESSOR")
print("=" * 40)

print("\nTraining Performance:")
print("MAE :", round(mean_absolute_error(y_reg_train, rf_train_pred), 4), "months")
print("RMSE:", round(np.sqrt(mean_squared_error(y_reg_train, rf_train_pred)), 4), "months")
print("R²  :", round(r2_score(y_reg_train, rf_train_pred), 4))

print("\nTest Performance:")
print("MAE :", round(mean_absolute_error(y_reg_test, rf_test_pred), 4), "months")
print("RMSE:", round(np.sqrt(mean_squared_error(y_reg_test, rf_test_pred)), 4), "months")
print("R²  :", round(r2_score(y_reg_test, rf_test_pred), 4))

RANDOM FOREST REGRESSOR

Training Performance:
MAE : 2.7719 months
RMSE: 5.4123 months
R²  : 0.9664

Test Performance:
MAE : 5.2795 months
RMSE: 7.9529 months
R²  : 0.8506


In [36]:
error_df = X_reg_test.copy()

error_df['Actual Delay Months'] = y_reg_test.values
error_df['Predicted Delay Months'] = rf_test_pred

error_df['Absolute Error'] = np.abs(
    error_df['Actual Delay Months'] -
    error_df['Predicted Delay Months']
)

error_df = error_df.sort_values(
    'Absolute Error',
    ascending=False
)

print("Largest prediction errors:")
display(
    error_df[
        [
            'Actual Delay Months',
            'Predicted Delay Months',
            'Absolute Error'
        ]
    ].head(10)
)

Largest prediction errors:


,Actual Delay Months,Predicted Delay Months,Absolute Error
45,-9.166667,20.509778,29.676444
131,56.733333,39.670778,17.062556
47,0.000000,10.452111,10.452111
43,0.000000,8.270889,8.270889
167,29.466667,36.840111,7.373444
80,9.033333,2.479111,6.554222
59,12.100000,5.641556,6.458444
77,6.900000,0.511556,6.388444
134,34.466667,28.659000,5.807667
78,25.933333,21.365222,4.568111


In [37]:
# Feature Importance - Random Forest Regressor

feature_names = (
    random_forest_reg_pipeline
    .named_steps['preprocessor']
    .get_feature_names_out()
)

rf_model = random_forest_reg_pipeline.named_steps['model']

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance_df.head(20))

,Feature,Importance
8,num__Time Elapsed %,0.614876
7,num__Time Elapsed Days,0.241378
0,num__Original Cost\n(in cr.),0.028308
4,num__Remaining Budget\n(in cr.),0.017751
6,num__Planned Duration Days,0.012949
1,num__Expenditure\n(in cr.),0.010899
5,num__Progress Gap,0.009181
38,cat__Implementing Agency_Department of water r...,0.008367
2,num__Physical Progress\n(in %),0.008191
23,cat__Line Ministry_Department of Water Resourc...,0.005909


In [38]:
# ==========================================
# NirmaanAI - Historical Regression Features
# ==========================================

# Features that do NOT depend on today's date
historical_reg_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

# Keep only projects with known historical delay
historical_reg_df = df[
    df['Schedule Delay Months'].notna()
].copy()

X_hist_reg = historical_reg_df[historical_reg_features].copy()
y_hist_reg = historical_reg_df['Schedule Delay Months'].copy()

print("Historical projects:", len(historical_reg_df))
print("Features:", len(historical_reg_features))
print("X shape:", X_hist_reg.shape)
print("y shape:", y_hist_reg.shape)

Historical projects: 134
Features: 10
X shape: (134, 10)
y shape: (134,)


In [39]:
# ==========================================
# Preprocessing for Historical Regression
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_hist_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days'
]

categorical_hist_features = [
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

historical_preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_hist_features),
    ('cat', categorical_transformer, categorical_hist_features)
])

print("Historical preprocessing pipeline created successfully!")

Historical preprocessing pipeline created successfully!


In [40]:
# ==========================================
# Train Corrected Historical Regression Models
# ==========================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

# Linear Regression
historical_linear_pipeline = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', LinearRegression())
])

# Random Forest
historical_rf_pipeline = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# XGBoost
historical_xgb_pipeline = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42,
        n_jobs=-1
    ))
])

print("All three corrected regression pipelines created successfully!")

All three corrected regression pipelines created successfully!


In [41]:
# ==========================================
# Train-Test Split for Historical Regression
# ==========================================

from sklearn.model_selection import train_test_split

X_hist_train, X_hist_test, y_hist_train, y_hist_test = train_test_split(
    X_hist_reg,
    y_hist_reg,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_hist_train.shape[0])
print("Testing samples:", X_hist_test.shape[0])
print("Training target mean:", round(y_hist_train.mean(), 2))
print("Testing target mean:", round(y_hist_test.mean(), 2))

Training samples: 107
Testing samples: 27
Training target mean: 26.55
Testing target mean: 17.86


In [42]:
# ==========================================
# Train Historical Regression Models
# ==========================================

historical_linear_pipeline.fit(X_hist_train, y_hist_train)
historical_rf_pipeline.fit(X_hist_train, y_hist_train)
historical_xgb_pipeline.fit(X_hist_train, y_hist_train)

print("All three historical regression models trained successfully!")

All three historical regression models trained successfully!


In [43]:
# ==========================================
# 5-Fold Cross-Validation Comparison
# Historical Regression Models
# ==========================================

from sklearn.model_selection import KFold, cross_validate
import pandas as pd
import numpy as np

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    'Linear Regression': historical_linear_pipeline,
    'Random Forest': historical_rf_pipeline,
    'XGBoost': historical_xgb_pipeline
}

cv_results = []

for model_name, model in models.items():

    scores = cross_validate(
        model,
        X_hist_train,
        y_hist_train,
        cv=kf,
        scoring={
            'MAE': 'neg_mean_absolute_error',
            'RMSE': 'neg_root_mean_squared_error',
            'R2': 'r2'
        },
        n_jobs=-1
    )

    cv_results.append({
        'Model': model_name,
        'Mean MAE (months)': -scores['test_MAE'].mean(),
        'Mean RMSE (months)': -scores['test_RMSE'].mean(),
        'Mean R²': scores['test_R2'].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by='Mean MAE (months)',
    ascending=True
).reset_index(drop=True)

display(cv_results_df.round(4))

,Model,Mean MAE (months),Mean RMSE (months),Mean R²
0,XGBoost,17.7242,24.5221,0.2862
1,Random Forest,17.7975,24.2754,0.3120
2,Linear Regression,23.9971,34.1874,-0.4623


In [44]:
# ==========================================
# Hyperparameter Tuning - Historical RF
# ==========================================

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

rf_param_grid_hist = {
    'model__n_estimators': randint(200, 1000),
    'model__max_depth': [None, 5, 8, 10, 15, 20, 25, 30],
    'model__min_samples_split': randint(2, 12),
    'model__min_samples_leaf': randint(1, 6),
    'model__max_features': ['sqrt', 'log2', 0.5, 0.7, 1.0]
}

rf_random_search_hist = RandomizedSearchCV(
    estimator=historical_rf_pipeline,
    param_distributions=rf_param_grid_hist,
    n_iter=60,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random_search_hist.fit(
    X_hist_train,
    y_hist_train
)

print("Best RF parameters:")
print(rf_random_search_hist.best_params_)

print(
    "\nBest CV MAE:",
    round(-rf_random_search_hist.best_score_, 4),
    "months"
)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best RF parameters:
{'model__max_depth': 8, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 4, 'model__n_estimators': 766}

Best CV MAE: 17.9943 months


In [45]:
# ==========================================
# Hyperparameter Tuning - Historical XGBoost
# ==========================================

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_param_grid_hist = {
    'model__n_estimators': randint(200, 1000),
    'model__max_depth': [2, 3, 4, 5, 6, 8],
    'model__learning_rate': uniform(0.01, 0.15),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 8),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(0.5, 2)
}

xgb_random_search_hist = RandomizedSearchCV(
    estimator=historical_xgb_pipeline,
    param_distributions=xgb_param_grid_hist,
    n_iter=60,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_random_search_hist.fit(
    X_hist_train,
    y_hist_train
)

print("Best XGBoost parameters:")
print(xgb_random_search_hist.best_params_)

print(
    "\nBest CV MAE:",
    round(-xgb_random_search_hist.best_score_, 4),
    "months"
)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best XGBoost parameters:
{'model__colsample_bytree': np.float64(0.6372411071223597), 'model__learning_rate': np.float64(0.144582363692999), 'model__max_depth': 6, 'model__min_child_weight': 3, 'model__n_estimators': 208, 'model__reg_alpha': np.float64(0.2788713525921819), 'model__reg_lambda': np.float64(1.9007156599455426), 'model__subsample': np.float64(0.9386644568953224)}

Best CV MAE: 17.7176 months


In [46]:
# ==========================================
# Final Test Set Comparison
# ==========================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

final_models = {
    'Tuned XGBoost': xgb_random_search_hist.best_estimator_,
    'Baseline Random Forest': historical_rf_pipeline
}

final_test_results = []

for model_name, model in final_models.items():

    predictions = model.predict(X_hist_test)

    mae = mean_absolute_error(y_hist_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_hist_test, predictions))
    r2 = r2_score(y_hist_test, predictions)

    final_test_results.append({
        'Model': model_name,
        'Test MAE (months)': mae,
        'Test RMSE (months)': rmse,
        'Test R²': r2
    })

final_test_results_df = pd.DataFrame(final_test_results)

display(final_test_results_df.round(4))

,Model,Test MAE (months),Test RMSE (months),Test R²
0,Tuned XGBoost,16.7148,21.3924,-0.0811
1,Baseline Random Forest,12.8143,17.3366,0.2900


In [47]:
# ==========================================
# Random Forest Regression Error Analysis
# ==========================================

# Predictions from the selected baseline RF
rf_hist_test_pred = historical_rf_pipeline.predict(X_hist_test)

error_df = X_hist_test.copy()

error_df['Actual Delay Months'] = y_hist_test.values
error_df['Predicted Delay Months'] = rf_hist_test_pred

error_df['Absolute Error'] = np.abs(
    error_df['Actual Delay Months'] -
    error_df['Predicted Delay Months']
)

error_df = error_df.sort_values(
    'Absolute Error',
    ascending=False
)

display(
    error_df[
        [
            'Actual Delay Months',
            'Predicted Delay Months',
            'Absolute Error'
        ]
    ].head(10).round(2)
)

,Actual Delay Months,Predicted Delay Months,Absolute Error
144,64.37,23.07,41.29
106,70.27,36.29,33.98
45,-9.17,20.10,29.26
131,56.73,28.38,28.35
117,-12.13,14.82,26.95
47,0.00,26.46,26.46
116,0.00,23.91,23.91
139,1.77,25.31,23.55
134,34.47,17.10,17.36
135,17.20,4.17,13.03


In [48]:
# ==========================================
# Random Forest Feature Importance
# ==========================================

# Get the fitted preprocessing step
fitted_preprocessor = historical_rf_pipeline.named_steps['preprocessor']

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Get RF importance
rf_model = historical_rf_pipeline.named_steps['model']

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    'Importance',
    ascending=False
).reset_index(drop=True)

display(
    feature_importance_df.head(15).round(4)
)


,Feature,Importance
0,num__Physical Progress\n(in %),0.2125
1,num__Original Cost\n(in cr.),0.1225
2,num__Progress Gap,0.1207
3,num__Planned Duration Days,0.1181
4,num__Expenditure\n(in cr.),0.0731
5,num__Remaining Budget\n(in cr.),0.0712
6,num__Expenditure %,0.0591
7,cat__Implementing Agency_Department of water r...,0.0512
8,cat__Sector Name_Water Resources,0.0498
9,cat__Implementing Agency_MoRTH,0.0434


In [49]:
# ==========================================
# Train Final Random Forest Regression Model
# ==========================================

final_rf_regression = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Train on all historical projects
final_rf_regression.fit(X_hist_reg, y_hist_reg)

print("Final Random Forest Regression model trained successfully!")
print("Training samples:", len(X_hist_reg))

Final Random Forest Regression model trained successfully!
Training samples: 134


In [50]:
# ==========================================
# Sanity Test - Final Regression Model
# ==========================================

sample_predictions = final_rf_regression.predict(
    X_hist_reg.head(5)
)

for i, prediction in enumerate(sample_predictions):
    print(f"Project {i+1}: Predicted Schedule Delay = {prediction:.2f} months")

Project 1: Predicted Schedule Delay = 20.57 months
Project 2: Predicted Schedule Delay = 37.51 months
Project 3: Predicted Schedule Delay = 34.92 months
Project 4: Predicted Schedule Delay = 27.60 months
Project 5: Predicted Schedule Delay = 10.59 months


In [51]:
# Find ongoing projects where revised commissioning date is missing
ongoing_projects = df[
    df['Revised\nDate of Commissioning'].isna()
].copy()

print("Ongoing / unknown projects:", len(ongoing_projects))

display(
    ongoing_projects[
        [
            'Sr. No.',
            'Project Name',
            'Original\nDate of Commissioning',
            'Sanction Date',
            'Physical Progress\n(in %)',
            'Original Cost\n(in cr.)',
            'Expenditure\n(in cr.)'
        ]
    ].head(10)
)

Ongoing / unknown projects: 48


,Sr. No.,Project Name,Original\nDate of Commissioning,Sanction Date,Physical Progress\n(in %),Original Cost\n(in cr.),Expenditure\n(in cr.)
2,3,AMAL GONDEGAON GHATROHANA EXPN,2029-03-31,2024-09-10,3,669,2.95
5,6,AMLG GAURI PAUNI EXPANSION OC,2030-03-31,2024-04-26,4,1225,92.02
6,7,BALLARPUR NORTH WEST OC MINE,2029-03-31,2022-12-16,5,361,26.97
7,8,BHATADI EXPANSION OC,2028-03-31,2020-02-11,34,581,112.17
8,9,DHUPTALA OC [SASTI UG TO OC ],2028-03-31,2019-02-12,19,711,174.66
9,10,DINESH[MAKARDHOKRA-III] OC,2027-03-31,2020-02-11,41,823,792.00
10,11,GADEGAON OC 3 MTY,2030-03-31,2023-10-28,2,497,95.69
11,12,GAURIDEEP/GAURI CENTRAL OCP 7.,2030-03-31,2023-08-08,2,973,1.55
12,13,GHONSA EXPANSION DEEP OC MINE,2029-03-31,2021-03-23,13,185,15.72
14,15,KOLARPIMPRI EXPANSION OC [2.5,2030-03-31,2024-06-05,0,630,0.61


In [52]:
# ==========================================
# Select one ongoing project for prediction
# ==========================================

test_project = ongoing_projects[
    ongoing_projects['Sr. No.'] == 9
].copy()

display(test_project)

,Sr. No.,Sector Name,Line Ministry,Implementing Agency,Project Code,Project Name,Original Cost\n(in cr.),Revised Cost\n(in cr.),Expenditure\n(in cr.),Physical Progress\n(in %),...,Progress Deviation,Progress Status,Remaining Project Time Days,Remaining Project Time Months,Planned Duration Years,Time Elapsed Years,Schedule Delay Years,Remaining Project Time Years,Delay Risk,Delay Risk Label
8,9,Coal,Ministry of Coal,WCL - CIL,400353,DHUPTALA OC [SASTI UG TO OC ],711,NaN,174.66,19,...,-63.818591,Severely Behind,NaN,NaN,9.13,7.56,NaN,NaN,NaN,NaN


In [53]:
# ==========================================
# Prepare current project for ML prediction
# ==========================================

test_project_ml = test_project[historical_reg_features].copy()

display(test_project_ml)

,Original Cost\n(in cr.),Expenditure\n(in cr.),Physical Progress\n(in %),Expenditure %,Remaining Budget\n(in cr.),Progress Gap,Planned Duration Days,Sector Name,Line Ministry,Implementing Agency
8,711,174.66,19,24.565401,536.34,5.565401,3335.0,Coal,Ministry of Coal,WCL - CIL


In [54]:
# ==========================================
# Predict Schedule Delay for Test Project
# ==========================================

predicted_delay_months = final_rf_regression.predict(test_project_ml)[0]

print(f"Predicted Schedule Delay: {predicted_delay_months:.2f} months")

Predicted Schedule Delay: 8.05 months


In [56]:
# ==========================================
# Convert Predicted Delay to Completion Date
# ==========================================

import pandas as pd

original_completion_date = pd.to_datetime(
    test_project['Original\nDate of Commissioning'].iloc[0]
)

# Split prediction into whole months + fractional month
whole_months = int(predicted_delay_months)
fractional_month = predicted_delay_months - whole_months

# Add whole months
predicted_completion_date = (
    original_completion_date
    + pd.DateOffset(months=whole_months)
)

# Convert fractional month approximately to days
extra_days = round(fractional_month * 30.44)

predicted_completion_date = (
    predicted_completion_date
    + pd.Timedelta(days=extra_days)
)

print("Original Planned Completion:",
      original_completion_date.strftime("%d %B %Y"))

print("Predicted Delay:",
      f"{predicted_delay_months:.2f} months")

print("Predicted Completion Date:",
      predicted_completion_date.strftime("%d %B %Y"))

Original Planned Completion: 31 March 2028
Predicted Delay: 8.05 months
Predicted Completion Date: 02 December 2028


In [57]:
import joblib

joblib.dump(
    final_rf_regression,
    'NirmaanAI_RF_Delay_Regression.pkl'
)

print("Final regression model saved successfully!")

Final regression model saved successfully!


In [58]:
# ==========================================
# Historical-Safe Classification Dataset
# ==========================================

historical_class_df = df[df['Delay Risk'].notna()].copy()

X_hist_class = historical_class_df[historical_reg_features].copy()
y_hist_class = historical_class_df['Delay Risk'].copy()

print("Historical classification projects:", len(X_hist_class))
print("Features:", len(X_hist_class.columns))
print("Delayed:", (y_hist_class == 1).sum())
print("Not Delayed:", (y_hist_class == 0).sum())

Historical classification projects: 134
Features: 10
Delayed: 105
Not Delayed: 29


In [59]:
# ==========================================
# Train-Test Split for Historical Classifier
# ==========================================

X_hist_class_train, X_hist_class_test, y_hist_class_train, y_hist_class_test = train_test_split(
    X_hist_class,
    y_hist_class,
    test_size=0.20,
    random_state=42,
    stratify=y_hist_class
)

print("Training samples:", len(X_hist_class_train))
print("Testing samples:", len(X_hist_class_test))
print("Training delayed:", (y_hist_class_train == 1).sum())
print("Training not delayed:", (y_hist_class_train == 0).sum())
print("Testing delayed:", (y_hist_class_test == 1).sum())
print("Testing not delayed:", (y_hist_class_test == 0).sum())

Training samples: 107
Testing samples: 27
Training delayed: 84
Training not delayed: 23
Testing delayed: 21
Testing not delayed: 6


In [60]:
# ==========================================
# Historical-Safe XGBoost Classifier
# ==========================================

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# Account for class imbalance
negative_count = (y_hist_class_train == 0).sum()
positive_count = (y_hist_class_train == 1).sum()

scale_pos_weight_hist = negative_count / positive_count

historical_xgb_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_hist,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

historical_xgb_class_pipeline = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', historical_xgb_classifier)
])

print("Historical-safe XGBoost classifier pipeline created successfully!")
print("Scale positive weight:", round(scale_pos_weight_hist, 2))

Historical-safe XGBoost classifier pipeline created successfully!
Scale positive weight: 0.27


In [61]:
# ==========================================
# Train Historical-Safe XGBoost Classifier
# ==========================================

historical_xgb_class_pipeline.fit(
    X_hist_class_train,
    y_hist_class_train
)

print("Historical-safe XGBoost classifier trained successfully!")

Historical-safe XGBoost classifier trained successfully!


In [62]:
# ==========================================
# Evaluate Historical-Safe XGBoost Classifier
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_hist_class_pred = historical_xgb_class_pipeline.predict(
    X_hist_class_test
)

# Probabilities for ROC-AUC
y_hist_class_prob = historical_xgb_class_pipeline.predict_proba(
    X_hist_class_test
)[:, 1]

# Metrics
accuracy = accuracy_score(y_hist_class_test, y_hist_class_pred)
precision = precision_score(y_hist_class_test, y_hist_class_pred)
recall = recall_score(y_hist_class_test, y_hist_class_pred)
f1 = f1_score(y_hist_class_test, y_hist_class_pred)
roc_auc = roc_auc_score(y_hist_class_test, y_hist_class_prob)

print("Historical-Safe XGBoost Classification Results")
print("=" * 50)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_hist_class_test, y_hist_class_pred))

print("\nClassification Report:")
print(classification_report(
    y_hist_class_test,
    y_hist_class_pred,
    target_names=["Not Delayed", "Delayed"]
))

Historical-Safe XGBoost Classification Results
Accuracy : 0.7407
Precision: 0.8500
Recall   : 0.8095
F1 Score : 0.8293
ROC-AUC  : 0.8333

Confusion Matrix:
[[ 3  3]
 [ 4 17]]

Classification Report:
              precision    recall  f1-score   support

 Not Delayed       0.43      0.50      0.46         6
     Delayed       0.85      0.81      0.83        21

    accuracy                           0.74        27
   macro avg       0.64      0.65      0.65        27
weighted avg       0.76      0.74      0.75        27



In [63]:
# ==========================================
# Train Final Historical-Safe Classifier
# ==========================================

final_xgb_classifier = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_hist,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

# Train on all 134 labeled historical projects
final_xgb_classifier.fit(
    X_hist_class,
    y_hist_class
)

print("Final XGBoost classifier trained successfully!")
print("Training samples:", len(X_hist_class))

Final XGBoost classifier trained successfully!
Training samples: 134


In [64]:
# Sanity test: predict delay risk for the selected ongoing project

delay_prediction = final_xgb_classifier.predict(test_project[historical_reg_features])[0]
delay_probability = final_xgb_classifier.predict_proba(
    test_project[historical_reg_features]
)[0, 1]

print("Project:", test_project['Project Name'].iloc[0])
print("Predicted Delay Risk:", "Delayed" if delay_prediction == 1 else "Not Delayed")
print("Delay Probability:", round(delay_probability * 100, 2), "%")

Project: DHUPTALA OC [SASTI UG TO OC ]
Predicted Delay Risk: Not Delayed
Delay Probability: 30.15 %


In [65]:
# Compare classifier and regression predictions for DHUPTALA

project_features = test_project[historical_reg_features]

# 1. Classification
delay_class = final_xgb_classifier.predict(project_features)[0]
delay_probability = final_xgb_classifier.predict_proba(project_features)[0, 1]

# 2. Regression
predicted_delay_months = final_rf_regression.predict(project_features)[0]

print("PROJECT:", test_project['Project Name'].iloc[0])
print("-" * 50)

print("CLASSIFICATION")
print("Predicted Class:", "Delayed" if delay_class == 1 else "Not Delayed")
print("Probability of Delay:", round(delay_probability * 100, 2), "%")

print("\nREGRESSION")
print("Predicted Schedule Delay:", round(predicted_delay_months, 2), "months")

print("\nCURRENT PROJECT STATE")
print("Physical Progress:", test_project['Physical Progress\n(in %)'].iloc[0], "%")
print("Expenditure %:", round(test_project['Expenditure %'].iloc[0], 2), "%")
print("Progress Deviation:", round(test_project['Progress Deviation'].iloc[0], 2))
print("Progress Status:", test_project['Progress Status'].iloc[0])

PROJECT: DHUPTALA OC [SASTI UG TO OC ]
--------------------------------------------------
CLASSIFICATION
Predicted Class: Not Delayed
Probability of Delay: 30.15 %

REGRESSION
Predicted Schedule Delay: 8.05 months

CURRENT PROJECT STATE
Physical Progress: 19 %
Expenditure %: 24.57 %
Progress Deviation: -63.82
Progress Status: Severely Behind


In [66]:
# Check the project's actual ML input values

print("DHUPTALA — FEATURES USED BY THE MODEL")
print("=" * 60)

for feature in historical_reg_features:
    print(f"{feature}: {test_project[feature].iloc[0]}")

DHUPTALA — FEATURES USED BY THE MODEL
Original Cost
(in cr.): 711
Expenditure
(in cr.): 174.66
Physical Progress
(in %): 19
Expenditure %: 24.565400843881857
Remaining Budget
(in cr.): 536.34
Progress Gap: 5.565400843881857
Planned Duration Days: 3335.0
Sector Name: Coal
Line Ministry: Ministry of Coal
Implementing Agency: WCL - CIL


In [67]:
print("TEMPORAL / DATE FEATURE AUDIT")
print("=" * 60)

date_columns = [
    'Sanction Date',
    'Original\nDate of Commissioning',
    'Revised\nDate of Commissioning'
]

for col in date_columns:
    print(f"\n{col}")
    print("Non-null:", df[col].notna().sum())
    print("Missing :", df[col].isna().sum())
    print("Min     :", df[col].min())
    print("Max     :", df[col].max())

print("\n" + "=" * 60)
print("Dataset shape:", df.shape)
print("Rows per project:", df['Project Code'].nunique())

TEMPORAL / DATE FEATURE AUDIT

Sanction Date
Non-null: 182
Missing : 0
Min     : 1996-04-01
Max     : 2026-03-12

Original
Date of Commissioning
Non-null: 181
Missing : 1


TypeError: '<=' not supported between instances of 'str' and 'float'

In [68]:
# Temporal audit — safely convert dates only for this check

print("TEMPORAL / DATE FEATURE AUDIT")
print("=" * 60)

date_columns = [
    'Sanction Date',
    'Original\nDate of Commissioning',
    'Revised\nDate of Commissioning'
]

for col in date_columns:
    temp_dates = pd.to_datetime(df[col], errors='coerce')

    print(f"\n{col}")
    print("Non-null:", temp_dates.notna().sum())
    print("Missing :", temp_dates.isna().sum())
    print("Min     :", temp_dates.min())
    print("Max     :", temp_dates.max())

print("\n" + "=" * 60)
print("Dataset shape:", df.shape)
print("Unique projects:", df['Project Code'].nunique())

TEMPORAL / DATE FEATURE AUDIT

Sanction Date
Non-null: 182
Missing : 0
Min     : 1996-04-01 00:00:00
Max     : 2026-03-12 00:00:00

Original
Date of Commissioning
Non-null: 181
Missing : 1
Min     : 2019-06-02 00:00:00
Max     : 2032-03-31 00:00:00

Revised
Date of Commissioning
Non-null: 134
Missing : 48
Min     : 2019-03-31 00:00:00
Max     : 2029-12-31 00:00:00

Dataset shape: (182, 40)
Unique projects: 182


In [69]:
print("ALL FEATURES IN NirmaanAI DATASET")
print("=" * 60)

for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

ALL FEATURES IN NirmaanAI DATASET
 1. Sr. No.
 2. Sector Name
 3. Line Ministry
 4. Implementing Agency
 5. Project Code
 6. Project Name
 7. Original Cost
(in cr.)
 8. Revised Cost
(in cr.)
 9. Expenditure
(in cr.)
10. Physical Progress
(in %)
11. Original
Date of Commissioning
12. Revised
Date of Commissioning
13. Sanction Date
14. Schedule Delay Days
15. Schedule Delay Months
16. Schedule Status
17. Cost Overrun
18. Cost Overrun %
19. Cost Status
20. Expenditure %
21. Remaining Budget
(in cr.)
22. Project Stage
23. Progress Gap
24. Expenditure Efficiency Status
25. Planned Duration Days
26. Planned Duration Months
27. Time Elapsed Days
28. Time Elapsed Months
29. Time Elapsed %
30. Expected Progress %
31. Progress Deviation
32. Progress Status
33. Remaining Project Time Days
34. Remaining Project Time Months
35. Planned Duration Years
36. Time Elapsed Years
37. Schedule Delay Years
38. Remaining Project Time Years
39. Delay Risk
40. Delay Risk Label


In [70]:
# FINAL NIRMAANAI REGRESSION MODEL

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import joblib

final_ml_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

# Historical projects with known schedule delay
final_reg_df = df[df['Schedule Delay Months'].notna()].copy()

X_final = final_reg_df[final_ml_features]
y_final = final_reg_df['Schedule Delay Months']

# Train final model on all available labeled historical projects
final_rf_model = Pipeline([
    ('preprocessor', historical_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

final_rf_model.fit(X_final, y_final)

print("Final NirmaanAI regression model trained successfully!")
print("Training samples:", len(X_final))
print("Features:", len(final_ml_features))

Final NirmaanAI regression model trained successfully!
Training samples: 134
Features: 10


In [71]:
def predict_project_delay(project_data):

    # Convert input into DataFrame
    input_df = pd.DataFrame([project_data])

    # Predict delay in months
    predicted_delay = final_rf_model.predict(input_df)[0]

    # Original planned completion date
    original_completion = pd.to_datetime(
        input_df['Original\nDate of Commissioning'].iloc[0]
    )

    # Convert predicted months into a date
    whole_months = int(predicted_delay)
    fractional_month = predicted_delay - whole_months

    estimated_completion = (
        original_completion +
        pd.DateOffset(months=whole_months)
    )

    extra_days = round(fractional_month * 30.44)
    estimated_completion += pd.Timedelta(days=extra_days)

    return {
        "predicted_delay_months": round(predicted_delay, 2),
        "estimated_completion_date": estimated_completion.strftime("%d %B %Y")
    }

In [73]:
def predict_project_delay(project_data, original_completion_date):

    # Convert input into DataFrame
    input_df = pd.DataFrame([project_data])

    # Predict delay
    predicted_delay = final_rf_model.predict(input_df)[0]

    # Convert predicted delay into completion date
    original_completion = pd.to_datetime(original_completion_date)

    whole_months = int(predicted_delay)
    fractional_month = predicted_delay - whole_months

    estimated_completion = (
        original_completion +
        pd.DateOffset(months=whole_months)
    )

    extra_days = round(fractional_month * 30.44)
    estimated_completion += pd.Timedelta(days=extra_days)

    return {
        "predicted_delay_months": round(predicted_delay, 2),
        "estimated_completion_date": estimated_completion.strftime("%d %B %Y")
    }

In [74]:
project_input = test_project[[
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]].iloc[0].to_dict()

original_completion_date = test_project[
    'Original\nDate of Commissioning'
].iloc[0]

prediction = predict_project_delay(
    project_input,
    original_completion_date
)

print("NirmaanAI Prediction")
print("-" * 30)
print("Project:", test_project['Project Name'].iloc[0])
print("Predicted Delay:", prediction["predicted_delay_months"], "months")
print("Estimated Completion:", prediction["estimated_completion_date"])

NirmaanAI Prediction
------------------------------
Project: DHUPTALA OC [SASTI UG TO OC ]
Predicted Delay: 8.05 months
Estimated Completion: 02 December 2028


In [75]:
def assess_project_health(project):

    physical_progress = project['Physical Progress\n(in %)']
    progress_deviation = project['Progress Deviation']
    expenditure_efficiency = project['Expenditure Efficiency Status']

    # Determine current project health
    if progress_deviation <= -30:
        health = "Severely Behind"
        priority = "Critical"
    elif progress_deviation <= -15:
        health = "Behind Schedule"
        priority = "High"
    elif progress_deviation <= 10:
        health = "On Track"
        priority = "Medium"
    else:
        health = "Ahead"
        priority = "Low"

    return {
        "physical_progress": round(physical_progress, 2),
        "progress_deviation": round(progress_deviation, 2),
        "expenditure_efficiency": expenditure_efficiency,
        "current_health": health,
        "priority": priority
    }

In [76]:
health = assess_project_health(test_project.iloc[0])

print("Current Project Health")
print("-" * 30)

for key, value in health.items():
    print(f"{key}: {value}")

Current Project Health
------------------------------
physical_progress: 19
progress_deviation: -63.82
expenditure_efficiency: Balanced
current_health: Severely Behind
priority: Critical


In [77]:
def nirmaanai_predict(project):

    # -----------------------------
    # 1. ML INPUTS
    # -----------------------------
    project_input = project[[
        'Original Cost\n(in cr.)',
        'Expenditure\n(in cr.)',
        'Physical Progress\n(in %)',
        'Expenditure %',
        'Remaining Budget\n(in cr.)',
        'Progress Gap',
        'Planned Duration Days',
        'Sector Name',
        'Line Ministry',
        'Implementing Agency'
    ]].to_dict()

    # -----------------------------
    # 2. DELAY PREDICTION
    # -----------------------------
    prediction = predict_project_delay(
        project_input,
        project['Original\nDate of Commissioning']
    )

    # -----------------------------
    # 3. CURRENT HEALTH
    # -----------------------------
    health = assess_project_health(project)

    # -----------------------------
    # 4. FINAL OUTPUT
    # -----------------------------
    return {
        "project_name": project['Project Name'],
        "predicted_delay_months": prediction["predicted_delay_months"],
        "estimated_completion_date": prediction["estimated_completion_date"],
        "physical_progress": health["physical_progress"],
        "progress_deviation": health["progress_deviation"],
        "expenditure_efficiency": health["expenditure_efficiency"],
        "current_health": health["current_health"],
        "priority": health["priority"]
    }

In [78]:
result = nirmaanai_predict(test_project.iloc[0])

print("NIRMAANAI PROJECT ASSESSMENT")
print("=" * 40)

for key, value in result.items():
    print(f"{key}: {value}")

NIRMAANAI PROJECT ASSESSMENT
project_name: DHUPTALA OC [SASTI UG TO OC ]
predicted_delay_months: 8.05
estimated_completion_date: 02 December 2028
physical_progress: 19
progress_deviation: -63.82
expenditure_efficiency: Balanced
current_health: Severely Behind
priority: Critical


In [79]:
import joblib

joblib.dump(
    final_rf_model,
    'NirmaanAI_Final_RF_Regression.pkl'
)

print("Final NirmaanAI regression model saved successfully!")

Final NirmaanAI regression model saved successfully!


In [80]:
import joblib

loaded_model = joblib.load('NirmaanAI_Final_RF_Regression.pkl')

print("Model loaded successfully!")
print(type(loaded_model))

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [82]:
final_ml_features = [
    'Original Cost\n(in cr.)',
    'Expenditure\n(in cr.)',
    'Physical Progress\n(in %)',
    'Expenditure %',
    'Remaining Budget\n(in cr.)',
    'Progress Gap',
    'Planned Duration Days',
    'Sector Name',
    'Line Ministry',
    'Implementing Agency'
]

joblib.dump(
    final_ml_features,
    'NirmaanAI_ML_Features.pkl'
)

print("Feature configuration saved successfully!")
print("Number of features:", len(final_ml_features))

Feature configuration saved successfully!
Number of features: 10
